In this notebook, I'll analyze the behavior from epilepsy patients. 

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
from typing import Dict, List, Tuple
import glob
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from scipy.spatial.distance import pdist, squareform
from scipy.stats import wilcoxon, spearmanr, pearsonr
from matplotlib.backends.backend_pdf import PdfPages
from mne.stats import permutation_cluster_1samp_test
from plot_config import set_plot_style

In [ ]:
import time, sys
from IPython.display import clear_output

def update_progress(progress):
    bar_length = 20
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
    if progress < 0:
        progress = 0
    if progress >= 1:
        progress = 1

    block = int(round(bar_length * progress))

    clear_output(wait = True)
    text = "Progress: [{0}] {1:.1f}%".format( "#" * block + "-" * (bar_length - block), progress * 100)
    print(text)

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
save_dir_plots = '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'
os.makedirs(save_dir_plots, exist_ok=True)


# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
def _gelman_standardize(values: pd.Series) -> pd.Series:
    arr = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    mu = np.nanmean(arr)
    sd = np.nanstd(arr)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.nan, index=values.index)
    return pd.Series((arr - mu) / (2.0 * sd), index=values.index)
    
def load_behavior_tables(load_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load behavior tables used by MV analyses.

    Returns
    -------
    (learn_df, combined_df)
    """
    learn_path = os.path.join(load_dir, "learn_df_RWH.csv")
    combined_path = os.path.join(load_dir, "full_df_RWH.csv")
    learn_df = pd.read_csv(learn_path)
    combined_df = pd.read_csv(combined_path)
    return learn_df, combined_df

def preprocess_learn_df(learn_df: pd.DataFrame) -> pd.DataFrame:
    """Replicates Subject-class learn_df preprocessing from mv_regression_utils.py."""
    df = learn_df.copy()

    if "gamble_rt" in df.columns and "gamble_rt_log" not in df.columns:
        with np.errstate(invalid="ignore"):
            df["gamble_rt_log"] = np.log(df["gamble_rt"])

    if {"rpe", "reward"}.issubset(df.columns):
        df["Q"] = df["rpe"] + df["reward"]

    if {"participant", "rpe"}.issubset(df.columns):
        df["prior_rpe"] = df.groupby("participant")["rpe"].shift(1)

    reversal_trials = [16, 33, 48, 65]
    if {"trials", "block"}.issubset(df.columns):
        def _tsr(row: pd.Series) -> float:
            blk = row.get("block", np.nan)
            tr = row.get("trials", np.nan)
            if not np.isfinite(blk) or not np.isfinite(tr):
                return np.nan
            if blk > 1:
                idx = int(blk - 2)
                if 0 <= idx < len(reversal_trials):
                    return float(np.mod(tr, reversal_trials[idx]))
            return float(tr)

        df["trials_since_reversal"] = df.apply(_tsr, axis=1)

    if "reward" in df.columns:
        df["preward"] = df["reward"].map({1: 0.5, 0: -0.5})

    if "gamble" in df.columns:
        df["pgamble"] = df["gamble"].map({1: 0.5, 0: -0.5})

    if "participant" in df.columns:
        for subj in df["participant"].dropna().unique():
            mask = df["participant"] == subj
            for src, dst in [
                ("Q", "zQ"),
                ("prior_rpe", "zprior_rpe"),
                ("rpe", "zrpe"),
                ("DPRIME", "zpm"),
                ("trials_since_reversal", "ztrials_since_reversal"),
            ]:
                if src in df.columns:
                    df.loc[mask, dst] = _gelman_standardize(df.loc[mask, src]).values

            if "gamble_rt_log" in df.columns and "reward" in df.columns:
                rt = pd.to_numeric(df.loc[mask, "gamble_rt_log"], errors="coerce").to_numpy(dtype=float)
                rt_min = np.nanmin(rt)
                rt_max = np.nanmax(rt)
                if np.isfinite(rt_min) and np.isfinite(rt_max) and (rt_max - rt_min) > 0:
                    rt_ev = 1.0 - ((rt - rt_min) / (rt_max - rt_min))
                    rt_rpe = pd.to_numeric(df.loc[mask, "reward"], errors="coerce").to_numpy(dtype=float) - rt_ev
                    df.loc[mask, "RT_rpe"] = rt_rpe
                    s = np.nanstd(rt_rpe)
                    if np.isfinite(s) and s > 0:
                        df.loc[mask, "zRT_rpe"] = (rt_rpe - np.nanmean(rt_rpe)) / (2.0 * s)

            if "rpe" in df.columns:
                vals = pd.to_numeric(df.loc[mask, "rpe"], errors="coerce")
                try:
                    df.loc[mask, "rpe_cat"] = pd.qcut(
                        vals,
                        q=3,
                        labels=["negative", "neutral", "positive"],
                        duplicates="drop",
                    )
                except Exception:
                    # Degenerate distributions can fail qcut.
                    df.loc[mask, "rpe_cat"] = np.nan

    return df


def preprocess_combined_df(combined_df: pd.DataFrame) -> pd.DataFrame:
    """Replicates Subject-class combined_df preprocessing from mv_regression_utils.py."""
    df = combined_df.copy()

    outcome_cols = ["hits", "misses", "correct_rejections", "false_alarms"]
    present_outcomes = [c for c in outcome_cols if c in df.columns]
    if len(present_outcomes) > 0:
        df["memory_outcome"] = df[present_outcomes].idxmax(axis=1)

    if "gamble_rt_log" not in df.columns and "gamble_rt_gamble" in df.columns:
        with np.errstate(invalid="ignore"):
            df["gamble_rt_log"] = np.log(df["gamble_rt_gamble"])

    if {"rpe", "reward_gamble"}.issubset(df.columns):
        df["Q"] = df["rpe"] + df["reward_gamble"]

    if {"participant", "rpe"}.issubset(df.columns):
        df["prior_rpe"] = df.groupby("participant")["rpe"].shift(1)

    reversal_trials = [16, 33, 48, 65]
    if {"trials_gamble", "block_gamble"}.issubset(df.columns):
        def _tsr(row: pd.Series) -> float:
            blk = row.get("block_gamble", np.nan)
            tr = row.get("trials_gamble", np.nan)
            if not np.isfinite(blk) or not np.isfinite(tr):
                return np.nan
            if blk > 1:
                idx = int(blk - 2)
                if 0 <= idx < len(reversal_trials):
                    return float(np.mod(tr, reversal_trials[idx]))
            return float(tr)

        df["trials_since_reversal"] = df.apply(_tsr, axis=1)

    if "hits" in df.columns:
        hits_bool = df["hits"].astype(bool)
        df["phit"] = np.where(hits_bool, 0.5, -0.5)

    if "reward_gamble" in df.columns:
        df["preward"] = df["reward_gamble"].map({1: 0.5, 0: -0.5})

    if "gamble_gamble" in df.columns:
        df["pgamble"] = df["gamble_gamble"].map({1: 0.5, 0: -0.5})

    if "participant" in df.columns:
        for subj in df["participant"].dropna().unique():
            mask = df["participant"] == subj
            for src, dst in [
                ("Q", "zQ"),
                ("prior_rpe", "zprior_rpe"),
                ("rpe", "zrpe"),
                ("DPRIME", "zpm"),
                ("trials_since_reversal", "ztrials_since_reversal"),
            ]:
                if src in df.columns:
                    df.loc[mask, dst] = _gelman_standardize(df.loc[mask, src]).values

            if "gamble_rt_log" in df.columns and "reward_gamble" in df.columns:
                rt = pd.to_numeric(df.loc[mask, "gamble_rt_log"], errors="coerce").to_numpy(dtype=float)
                rt_min = np.nanmin(rt)
                rt_max = np.nanmax(rt)
                if np.isfinite(rt_min) and np.isfinite(rt_max) and (rt_max - rt_min) > 0:
                    rt_ev = 1.0 - ((rt - rt_min) / (rt_max - rt_min))
                    rt_rpe = pd.to_numeric(df.loc[mask, "reward_gamble"], errors="coerce").to_numpy(dtype=float) - rt_ev
                    df.loc[mask, "RT_rpe"] = rt_rpe
                    s = np.nanstd(rt_rpe)
                    if np.isfinite(s) and s > 0:
                        df.loc[mask, "zRT_rpe"] = (rt_rpe - np.nanmean(rt_rpe)) / (2.0 * s)

            if "rpe" in df.columns:
                vals = pd.to_numeric(df.loc[mask, "rpe"], errors="coerce")
                try:
                    df.loc[mask, "rpe_cat"] = pd.qcut(
                        vals,
                        q=3,
                        labels=["neg", "neutral", "pos"],
                        duplicates="drop",
                    )
                except Exception:
                    df.loc[mask, "rpe_cat"] = np.nan

    return df

In [ ]:
learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

## Figure 3A

Example

## Figure 3B

Example

## Figure 3C

Schematic

## Figure 3D

MATLAB

## Figure 4A-C

MATLAB

## Figure 4D, S6

In [ ]:
def assert_required_columns(tbl: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
    missing = [c for c in required_cols if c not in tbl.columns]
    if missing:
        raise ValueError(f'{table_name} is missing required columns: {missing}')


def deterministic_inner_join(left: pd.DataFrame, right: pd.DataFrame, keys: List[str]) -> pd.DataFrame:
    out = left.merge(right, on=keys, how='inner')
    return out.sort_values(keys).reset_index(drop=True)


def _extract_corr(res) -> Tuple[float, float]:
    if hasattr(res, 'statistic') and hasattr(res, 'pvalue'):
        return float(res.statistic), float(res.pvalue)
    return float(res[0]), float(res[1])


def run_correlation_analysis(
    x: np.ndarray,
    y: np.ndarray,
    corr_type: str = 'spearman',
    run_secondary_pearson: bool = True,
) -> Dict[str, float]:
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask]
    y_valid = y[mask]
    n = int(len(x_valid))
    if n < 3:
        raise ValueError(f'Need at least 3 valid observations; got {n}.')

    rho_s, p_s = _extract_corr(spearmanr(x_valid, y_valid, nan_policy='omit'))
    if run_secondary_pearson or corr_type.lower() == 'pearson':
        r_p, p_p = _extract_corr(pearsonr(x_valid, y_valid))
    else:
        r_p, p_p = np.nan, np.nan

    corr_type_norm = corr_type.lower()
    if corr_type_norm == 'spearman':
        primary_r, primary_p = rho_s, p_s
    elif corr_type_norm == 'pearson':
        primary_r, primary_p = r_p, p_p
    else:
        raise ValueError(f"Unsupported corr_type: {corr_type}. Use 'spearman' or 'pearson'.")

    return {
        'n': n,
        'spearman_rho': rho_s,
        'spearman_p': p_s,
        'pearson_r': r_p,
        'pearson_p': p_p,
        'primary_corr_type': corr_type_norm,
        'primary_r': primary_r,
        'primary_p': primary_p,
    }


def _fit_and_plot_line(ax, x: np.ndarray, y: np.ndarray) -> None:
    if len(x) < 2 or len(np.unique(x)) < 2:
        return
    coeff = np.polyfit(x, y, 1)
    xfit = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    yfit = np.polyval(coeff, xfit)
    ax.plot(xfit, yfit, color='#222222', linewidth=1.2)


def load_mean_acc_hfa(
    subj_ids: List[str],
    base_dir: str,
    event: str,
    block: Tuple[float, float],
    acc_roi: str = 'ACC',
    hfa_band: str = 'hfa',
) -> pd.DataFrame:
    rows = []
    missing_elec_tables = []
    subjects_with_no_matching_csvs = []
    subjects_with_csvs_but_no_acc_hfa = []
    total_csv_files_found = 0
    block_str = f'time_averaged_df_time{block[0]}-{block[1]}'

    for subj_id in subj_ids:
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df'
        if not os.path.isfile(elec_path):
            missing_elec_tables.append(elec_path)
            continue

        elec_df = pd.read_csv(elec_path)
        region_col = None
        for candidate in ['salman_region', 'region']:
            if candidate in elec_df.columns:
                region_col = candidate
                break
        if region_col is None:
            raise ValueError(f'Could not identify ROI column for {subj_id}.')

        acc_labels = elec_df.loc[elec_df[region_col].astype(str) == acc_roi, 'label'].astype(str).tolist()
        if not acc_labels:
            subjects_with_no_matching_csvs.append(subj_id)
            continue

        subject_parts = []
        found_csv_count = 0

        for chan in acc_labels:
            chan_path = f'{filepath}/{event}/dfs/{chan}_{block_str}.csv'
            if not os.path.isfile(chan_path):
                continue

            found_csv_count += 1
            total_csv_files_found += 1
            chan_df = pd.read_csv(chan_path)
            required = ['tfr', 'fband']
            assert_required_columns(chan_df, required, f'{subj_id} {event} {chan}')

            if 'participant' not in chan_df.columns:
                chan_df['participant'] = subj_id

            if 'region' in chan_df.columns:
                chan_df = chan_df[chan_df['region'].astype(str) == acc_roi]

            chan_df = chan_df[chan_df['fband'].astype(str) == hfa_band]
            if len(chan_df) == 0:
                continue

            chan_df = chan_df[['participant', 'tfr']].copy()
            chan_df['participant'] = subj_id
            subject_parts.append(chan_df)

        if found_csv_count == 0:
            subjects_with_no_matching_csvs.append(subj_id)
            continue
        if not subject_parts:
            subjects_with_csvs_but_no_acc_hfa.append(subj_id)
            continue

        subj_tbl = pd.concat(subject_parts, ignore_index=True)
        rows.append({'Subject': subj_id, 'mean_hfa': float(subj_tbl['tfr'].mean())})

    out = pd.DataFrame(rows)
    if missing_elec_tables:
        print(f'Missing electrode tables for {len(missing_elec_tables)} subject(s).')
    print(f'Loaded {total_csv_files_found} ACC channel CSV(s) for event={event}, block={block}.')
    if subjects_with_no_matching_csvs:
        print(
            'Subjects with no matching ACC CSVs:',
            subjects_with_no_matching_csvs[:10],
            '...' if len(subjects_with_no_matching_csvs) > 10 else ''
        )
    if subjects_with_csvs_but_no_acc_hfa:
        print(
            'Subjects with ACC CSVs but no ACC/HFA rows after filtering:',
            subjects_with_csvs_but_no_acc_hfa[:10],
            '...' if len(subjects_with_csvs_but_no_acc_hfa) > 10 else ''
        )
    if out.empty:
        if total_csv_files_found == 0:
            raise ValueError(
                f'No ACC channel CSV files found for event={event}, block={block}. '
                f'Expected pattern: *_time_averaged_df_time{block[0]}-{block[1]}.csv'
            )
        raise ValueError(
            f'ACC channel CSVs were found for event={event}, block={block}, '
            f'but no rows survived ACC/HFA filtering.'
        )
    return out.sort_values('Subject').reset_index(drop=True)


def plot_subject_covariate_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    predictor_order: List[str],
    predictor_labels: Dict[str, str],
    output_pdf: Path,
    label_points: bool = True,
) -> None:
    n_panels = len(predictor_order)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    sns.set_style('white')

    for ax, predictor in zip(axes, predictor_order):
        panel_df = merged_df[np.isfinite(merged_df[predictor]) & np.isfinite(merged_df[target_col])].copy()
        ax.set_title(predictor_labels[predictor])
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[predictor].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)

        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, predictor), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        rr = results_df[results_df['predictor'] == predictor].iloc[0]
        ann = (
            f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, "
            f"p={rr['primary_p']:.3g}, n={int(rr['n'])}"
        )
        ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)
        ax.set_xlabel(predictor_labels[predictor])
        ax.set_ylabel(target_col)
        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    for ax in axes[n_panels:]:
        ax.axis('off')

    fig.suptitle(f'{target_col} vs subject-level ACC covariates', fontsize=14)
    fig.savefig(output_pdf, dpi=300)


In [ ]:
# ---- Config ----
reinstatement_metric = 'MeanAccuracy'  # or 'ReinstateEffect'
corr_type = 'spearman'  # 'spearman' or 'pearson'
run_secondary_pearson = True
label_points = True

acc_roi = 'ACC'
hfa_band = 'hfa'
feedback_event = 'feedback_start'
feedback_window: Tuple[float, float] = (0.0, 1.5)
cue_event = 'cue_on'
cue_window: Tuple[float, float] = (0.0, 1.0)

acc_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/crossdecoding/rpe_category/'
    'ACC_rpe_category_HFA_subject_reinstatement.csv'
)

analysis_output_root = Path(
    '/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/'
    'analysis_outputs/reinstatement_subject_covariates'
)
figure_output_root = Path(
    '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'
)
analysis_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)

analysis_prefix = f'rpe_category_HFA_{acc_roi}_{reinstatement_metric}_vs_subject_covariates'
paths: Dict[str, Path] = {
    'merged_csv': analysis_output_root / f'{analysis_prefix}_merged_subject_table.csv',
    'results_csv': analysis_output_root / f'{analysis_prefix}_correlation_results.csv',
    'summary_txt': analysis_output_root / f'{analysis_prefix}_summary.txt',
    'scatter_pdf': figure_output_root / 'acc_reinstatement_vs_subject_covariates_scatter.pdf',
}

predictor_order = [
    'ACC_NumElectrodes',
    'total_reward',
    'alpha',
    'beta',
    'ACC_mean_hfa_feedback',
    'ACC_mean_hfa_cue',
]
predictor_labels = {
    'ACC_NumElectrodes': 'ACC electrode count',
    'total_reward': 'Total reward',
    'alpha': 'Alpha',
    'beta': 'Beta',
    'ACC_mean_hfa_feedback': 'Mean ACC HFA during feedback',
    'ACC_mean_hfa_cue': 'Mean ACC HFA during cue',
}

print('ACC reinstatement CSV:', acc_reinstatement_csv)
print('Behavior CSV:', learn_df)
print('Analysis output root:', analysis_output_root)
print('Figure output root:', figure_output_root)
print('Feedback HFA:', feedback_event, feedback_window)
print('Cue HFA:', cue_event, cue_window)
print('N candidate subjects from demographics:', len(subj_ids))


In [ ]:
merged_subject_tbl = pd.read_csv(paths['merged_csv'])
results_df = pd.read_csv(paths['results_csv'])

In [ ]:
acc_metric_col = f'{acc_roi}_{reinstatement_metric}'

plot_subject_covariate_grid(
    merged_df=merged_subject_tbl,
    results_df=results_df,
    target_col=acc_metric_col,
    predictor_order=predictor_order,
    predictor_labels=predictor_labels,
    output_pdf=paths['scatter_pdf'],
    label_points=label_points,
)

## Figure 5A

Example

## Figure 5B

In [ ]:
from mv_regression_utils import (
    ROI_COLOR_MAP,
    OSCILLATION_EVENT_WINDOWS,
    OSCILLATION_FOOOF_DEFAULTS,
    plot_roi_oscillation_psd_summary,
    plot_roi_peak_frequency_histograms,
    plot_roi_property_deltas,
    plot_roi_property_summaries,
    run_roi_oscillation_comparison,
)

rois = ['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC', 'dlPFC']

event_windows = dict(OSCILLATION_EVENT_WINDOWS)
fooof_kwargs = dict(OSCILLATION_FOOOF_DEFAULTS)

In [ ]:
output_root = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots/roi_oscillation_comparison'
figure_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit_ROI_OscillationComparison'
table_dir = os.path.join(output_root, 'tables')

In [ ]:
# Load all
electrode_psd_df = pd.read_csv(os.path.join(table_dir, 'electrode_psd_df.csv'))
electrode_summary_df = pd.read_csv(os.path.join(table_dir, 'electrode_summary_df.csv'))
matched_pairs_df = pd.read_csv(os.path.join(table_dir, 'matched_pairs_df.csv'))
roi_psd_df = pd.read_csv(os.path.join(table_dir, 'roi_psd_df.csv'))
property_stats_df = pd.read_csv(os.path.join(table_dir, 'property_stats_df.csv'))
peak_stats_df = pd.read_csv(os.path.join(table_dir, 'peak_stats_df.csv'))
audit_df = pd.read_csv(os.path.join(table_dir, 'audit_df.csv'))


In [ ]:
fig_psd, _ = plot_roi_oscillation_psd_summary(
    roi_psd_df=roi_psd_df,
    rois=rois,
    colors=ROI_COLOR_MAP,
    xscale='log',
)
fig_psd.savefig(os.path.join(figure_dir, 'roi_corrected_psd.pdf'), dpi=300, bbox_inches='tight')
plt.show()

fig_psd_raw, _ = plot_roi_oscillation_psd_summary(
    roi_psd_df=roi_psd_df,
    rois=rois,
    colors=ROI_COLOR_MAP,
    spectrum='raw_z',
    xscale='log'
)
fig_psd_raw.savefig(os.path.join(figure_dir, 'roi_uncorrected_normalized_psd.pdf'), dpi=300, bbox_inches='tight')
plt.show()

fig_summary, _ = plot_roi_property_summaries(
    electrode_summary_df=electrode_summary_df,
    rois=rois,
    colors=ROI_COLOR_MAP,
)
fig_summary.savefig(os.path.join(figure_dir, 'roi_property_summaries.png'), dpi=300, bbox_inches='tight')
plt.show()


## Figure 5C

In [ ]:
lfp_repo = '/Users/salmanqasim/Documents/GitRepos/LFPAnalysis'
if lfp_repo not in sys.path:
    sys.path.append(lfp_repo)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
    
from bosc_theta_alpha_rpe_utils import (
    RPE_CATEGORY_ORDER,
    assign_exact_band,
    build_period_plot_df,
    build_time_plot_df,
    compute_period_stats,
    load_and_preprocess_behavior,
    load_group_bosc_outputs,
    plot_period_averaged_bosc,
    plot_time_resolved_bosc,
    run_subject_event_bosc,
)

sns.set_context('talk')
sns.set_style('white')

load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
ephys_root = f'{base_dir}/projects/guLab/Salman/EphysAnalyses'

In [ ]:
events = {
    'feedback_start': (0.0, 1.5),
    'cue_on': (0.0, 1.0),
}
rois = ['ACC', 'dmPFC', 'HPC']
bands = {
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
}

roi_colors = {
    'HPC': np.array([0.44314, 0.2902, 1.0]),
    'ACC': np.array([0.7098, 0.27059, 0.68235]),
    'dmPFC': np.array([0.40784, 0.01961, 0.0]),
}
rpe_palette = {
    'negative': '#E74C3C',
    'neutral': '#95A5A6',
    'positive': '#27AE60',
}

group_output_root = f'{ephys_root}/Plots/bosc_theta_alpha_rpe'
figure_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit_BOSC_RPE_ThetaAlpha'
table_dir = f'{group_output_root}/tables'

In [ ]:
period_df = pd.read_csv(f'{table_dir}/bosc_period_averaged_exact_band_all.csv')
freq_df = pd.read_csv(f'{table_dir}/bosc_time_averaged_frequency_all.csv')
period_plot_df = pd.read_csv(f'{table_dir}/bosc_period_plot_df.csv')

In [ ]:
from bosc_theta_alpha_rpe_utils import print_pairwise_rpe_ttests_period_plot

rois_band_plot = ['ACC', 'HPC']
events_band_plot = {'cue_on': events['cue_on']}

print_pairwise_rpe_ttests_period_plot(
    period_plot_df=period_plot_df,
    rois=rois_band_plot,
    events=events_band_plot,
    bands=bands,
)

# --- period-averaged BOSC by band (inline; edit freely) ---
event_names = list(events_band_plot.keys())
band_order = list(bands.keys())
n_rpe = len(RPE_CATEGORY_ORDER)

fig, axes = plt.subplots(
    len(rois_band_plot),
    len(event_names),
    figsize=(6, 3.4 * len(rois_band_plot)),
    sharey=True,
    dpi=300,
    squeeze=False,
)
axes = np.asarray(axes)

for i, roi in enumerate(rois_band_plot):
    for j, event in enumerate(event_names):
        ax = axes[i, j]
        plot_df = period_plot_df[
            (period_plot_df['region'] == roi)
            & (period_plot_df['event'] == event)
            & (period_plot_df['band'].isin(band_order))
        ].copy()
        raw_df = period_df[
            (period_df['region'] == roi)
            & (period_df['event'] == event)
            & (period_df['band'].isin(band_order))
        ].copy()
        n_elecs = int(raw_df['unique_elec_id'].nunique())
        n_subjects = int(plot_df['participant'].nunique())

        if plot_df.empty:
            ax.set_visible(False)
            continue

        sns.barplot(
            data=plot_df,
            x='band',
            y='prop_detect',
            hue='rpe_cat',
            order=band_order,
            hue_order=RPE_CATEGORY_ORDER,
            palette=rpe_palette,
            errorbar='se',
            ax=ax,
        )
        sns.stripplot(
            data=plot_df,
            x='band',
            y='prop_detect',
            hue='rpe_cat',
            order=band_order,
            hue_order=RPE_CATEGORY_ORDER,
            dodge=True,
            palette=rpe_palette,
            alpha=0.7,
            size=4,
            ax=ax,
        )

        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            ax.legend(handles[:n_rpe], labels[:n_rpe], frameon=False, title='RPE bin')

        ax.set_title(f'{roi} | {event} | n_subj={n_subjects}, n_elecs={n_elecs}')
        ax.set_xlabel('Band')
        ax.set_ylabel('Burst prevalence')
        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)
        ax.tick_params(
            axis='both', which='major', bottom=True, left=True, direction='out', length=4
        )

fig.suptitle('Period-averaged BOSC prevalence by RPE bin and band', y=1.02)
fig.tight_layout()

period_band_fig = fig
period_band_fig.savefig(
    f'/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots/period_averaged_band_comparison_rpe_bins.pdf',
    dpi=300,
    bbox_inches='tight',
)
plt.show()


## Figure 5D:

Schematic

In [ ]:
from mv_granger_utils import (
    load_behavior_tables,
    preprocess_learn_df,
    preprocess_combined_df,
    load_subject_epochs,
    run_or_load_subject_gc_rpe_df,
    load_gc_rpe_df_cache,
    merge_with_behavior_rpe,
    validate_smoke_outputs_rpe,
    generate_subject_plot_rpe,
    plot_smoke_rpe,
    plot_aggregate_rpe,
    fit_mixed_effects_rpe,
    plot_mixed_effects_rpe,
)

In [ ]:
# Analysis configuration
ANALYSIS = {
    'event': 'cue_on',
    'day': 'Day1',
    'tmin': 0.0,
    'tmax': 1.0,
    'target_sfreq': 50,
    'fmin': 4,
    'fmax': 8,
    'gc_n_lags': 10,
    'n_freqs': None,
    'min_trials': 5,
}

# Surrogate configuration (source-shuffle null)
USE_SURROGATES = True
N_SURROGATES_SMOKE = 100
N_SURROGATES_BATCH = 100
SURROGATE_TAIL = 'greater'   # one-sided
SURROGATE_RANDOM_STATE = 42
N_JOBS_SURROGATES_SMOKE = 12
N_JOBS_SURROGATES_BATCH = 12
SURROGATE_PARALLEL_BACKEND = 'loky'

# DataFrame cache configuration
GC_RPE_DF_CACHE_DIR = os.path.join(savepath, 'gc_df_cache_rpe')
OVERWRITE_GC_RPE_DF_CACHE = True

ROI_PAIRS = [
    ('HPC', 'ACC'),
    ('HPC', 'AMY'),
    ('HPC', 'dlPFC'),
    ('HPC', 'dmPFC'),
    ('HPC', 'OFC'),
    ('ACC', 'dmPFC'),
    ('ACC', 'dlPFC'),
    ('ACC', 'OFC'),
    ('ACC', 'AMY'),

]

RUN_SINGLE_SUBJECT_SMOKE = True
SMOKE_SUBJECT = 'UI002' if 'UI002' in subjects else subjects[0]
SMOKE_ROI_SOURCE, SMOKE_ROI_TARGET = ROI_PAIRS[0]

print('Smoke subject:', SMOKE_SUBJECT)
print('Smoke pair:', SMOKE_ROI_SOURCE, '->', SMOKE_ROI_TARGET)
print('Surrogates enabled:', USE_SURROGATES)
print('Smoke surrogates:', N_SURROGATES_SMOKE, '| Batch surrogates:', N_SURROGATES_BATCH)
print('GC RPE DF cache dir:', GC_RPE_DF_CACHE_DIR)
print('Overwrite cache:', OVERWRITE_GC_RPE_DF_CACHE)
rpe_behavior_df = combined_df if ANALYSIS['event'] == 'cue_on' else learn_df
print('RPE behavior source:', 'combined_df' if ANALYSIS['event'] == 'cue_on' else 'learn_df')

ANALYSIS_RESULTS = globals().get('ANALYSIS_RESULTS', {})
SMOKE_RESULTS = globals().get('SMOKE_RESULTS', {})


def _analysis_value_token(value):
    return str(value).replace('.', 'p').replace('-', 'm')


def _normalize_roi_pairs(pairs):
    return sorted([list(pair) for pair in pairs])


def _build_analysis_tag(analysis, roi_pairs):
    roi_pairs_norm = _normalize_roi_pairs(roi_pairs)
    roi_hash = hashlib.sha1(json.dumps(roi_pairs_norm, sort_keys=True).encode('utf-8')).hexdigest()[:10]
    band_tag_local = f"{analysis['fmin']}-{analysis['fmax']}Hz"
    return (
        f"{analysis['event']}_{analysis['day']}"
        f"_t{_analysis_value_token(analysis['tmin'])}-{_analysis_value_token(analysis['tmax'])}"
        f"_sf{_analysis_value_token(analysis['target_sfreq'])}"
        f"_{band_tag_local}"
        f"_lag{_analysis_value_token(analysis['gc_n_lags'])}"
        f"_roi{roi_hash}"
    )


analysis_tag = _build_analysis_tag(ANALYSIS, ROI_PAIRS)
fmin_tag = f"{ANALYSIS['fmin']}Hz"
band_tag = f"{ANALYSIS['fmin']}-{ANALYSIS['fmax']}Hz"
ANALYSIS_DIR = os.path.join(savepath, 'notebook_runs', analysis_tag)
os.makedirs(ANALYSIS_DIR, exist_ok=True)


def get_analysis_paths(tag=None, source_roi=None):
    tag = analysis_tag if tag is None else tag
    base_dir = os.path.join(savepath, 'notebook_runs', tag)
    os.makedirs(base_dir, exist_ok=True)
    paths = {
        'base_dir': base_dir,
        'all_gc_csv': os.path.join(base_dir, 'all_pairs_mvgc_rpe.csv'),
        'cache_log_csv': os.path.join(base_dir, 'cache_log_rpe.csv'),
        'merged_gc_csv': os.path.join(base_dir, 'merged_gc_rpe.csv'),
        'smoke_df_csv': os.path.join(base_dir, 'smoke_gc_rpe.csv'),
        'smoke_meta_json': os.path.join(base_dir, 'smoke_gc_rpe_meta.json'),
    }
    if source_roi is not None:
        paths['stats_csv'] = os.path.join(base_dir, f'mixed_model_gc_rpe_stats_{source_roi}.csv')
    return paths


analysis_paths = get_analysis_paths()
print('Analysis tag:', analysis_tag)
print('Analysis dir:', ANALYSIS_DIR)
print('Filename band tag:', band_tag)


In [ ]:
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if 'all_gc_df' in analysis_state:
    all_gc_df = analysis_state['all_gc_df'].copy()
else:
    all_gc_df = pd.read_csv(analysis_paths['all_gc_csv'])
    analysis_state['all_gc_df'] = all_gc_df.copy()
print('Loaded all_gc_df rows:', len(all_gc_df), '| from', analysis_paths['all_gc_csv'])


In [ ]:
# Export RPE outputs and behavior-merged files
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if len(all_gc_df) == 0:
    print('No GC rows to export.')
    merged_gc_df = pd.DataFrame()
elif 'merged_gc_df' in analysis_state:
    merged_gc_df = analysis_state['merged_gc_df'].copy()
    print('Loaded merged_gc_df from in-memory registry for', analysis_tag)
elif os.path.exists(analysis_paths['merged_gc_csv']):
    merged_gc_df = pd.read_csv(analysis_paths['merged_gc_csv'])
    print('Loaded merged_gc_df from analysis bundle:', analysis_paths['merged_gc_csv'])
else:
    all_gc_df.to_csv(analysis_paths['all_gc_csv'], index=False)
    print('Saved:', analysis_paths['all_gc_csv'])

    merged_rows = []
    for (roi_source, roi_target), pair_df in all_gc_df.groupby(['roi_source', 'roi_target']):
        merged = merge_with_behavior_rpe(pair_df.copy(), rpe_behavior_df, event=ANALYSIS['event'])
        merged_rows.append(merged)
        pair_file = os.path.join(analysis_paths['base_dir'], f'{roi_source}_{roi_target}_mvgc_rpe_merged.csv')
        merged.to_csv(pair_file, index=False)
    merged_gc_df = pd.concat(merged_rows, ignore_index=True) if len(merged_rows) > 0 else pd.DataFrame()
    merged_gc_df.to_csv(analysis_paths['merged_gc_csv'], index=False)
    print('Saved merged bundle:', analysis_paths['merged_gc_csv'])

analysis_state['all_gc_df'] = all_gc_df.copy()
analysis_state['merged_gc_df'] = merged_gc_df.copy()
if len(merged_gc_df) > 0:
    print('Merged rows:', len(merged_gc_df))
    display(merged_gc_df.head())


In [ ]:
# Aggregate plots + mixed-effects models (RPE)
# Load both events for consolidated left=feedback_start, right=cue_on plot (same axes).
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if 'merged_gc_df' in analysis_state:
    merged_gc_df = analysis_state['merged_gc_df'].copy()
else:
    merged_gc_df = pd.DataFrame()

event_specs = [
    ('feedback_start', 1.5),
    ('cue_on', 1.0),
]
event_frames = []
for ev_name, tmax in event_specs:
    analysis_ev = dict(ANALYSIS)
    analysis_ev['event'] = ev_name
    analysis_ev['tmax'] = tmax
    tag_ev = _build_analysis_tag(analysis_ev, ROI_PAIRS)
    path_ev = get_analysis_paths(tag=tag_ev)['merged_gc_csv']
    if os.path.exists(path_ev):
        df_ev = pd.read_csv(path_ev)
        df_ev['event'] = ev_name
        event_frames.append(df_ev)

combined_for_plot = pd.concat(event_frames, ignore_index=True) if len(event_frames) > 0 else pd.DataFrame()

if len(merged_gc_df) == 0 and len(combined_for_plot) == 0:
    print('No merged data for aggregate/model plotting.')
else:
    source_roi = 'HPC'
    source_paths = get_analysis_paths(source_roi=source_roi)
    agg_df = merged_gc_df[merged_gc_df['roi_source'] == source_roi].copy() if len(merged_gc_df) > 0 else pd.DataFrame()

    if len(agg_df) > 0:
        fig_agg = plot_aggregate_rpe(
            agg_df,
            source_roi=source_roi,
            savepath=source_paths['base_dir'],
            event=ANALYSIS['event'],
            fmin=ANALYSIS['fmin'],
            fmax=ANALYSIS['fmax'],
        )

        if fig_agg is not None:
            plt.show()

    # Event-split mixed-effects plot: left=feedback_start, right=cue_on, shared axes.
    if len(combined_for_plot) > 0:
        plot_df = combined_for_plot[combined_for_plot['roi_source'] == source_roi].copy()
        plot_df['target_roi'] = plot_df['roi_target']
        if ('gc_net_z' not in plot_df.columns) or (not plot_df['gc_net_z'].notna().any()):
            plot_df['gc_net_z'] = plot_df.groupby(['participant', 'roi_target', 'event'])['gc_net'].transform(
                lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
            )
        model_out_plot = {'model_df': plot_df, 'stats_df': pd.DataFrame()}
        fig_mixed = plot_mixed_effects_rpe(
            model_out_plot,
            source_roi=source_roi,
            savepath=source_paths['base_dir'],
            event=ANALYSIS['event'],
            fmin=ANALYSIS['fmin'],
            fmax=ANALYSIS['fmax'],
            ylim=(-2, 2),
        )

        if fig_mixed is not None:
            plt.show()
    else:
        # Fallback: single-event from merged_gc_df (right panel will show "no data" if plot expects event column).
        model_df = agg_df.copy()
        model_df['target_roi'] = model_df['roi_target']
        if len(model_df) > 0:
            if ('gc_net_z' not in model_df.columns) or (not model_df['gc_net_z'].notna().any()):
                model_df['gc_net_z'] = model_df.groupby(['participant', 'roi_target'])['gc_net'].transform(
                    lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
                )
            model_out = fit_mixed_effects_rpe(model_df, reference='neutral')
            fig_mixed = plot_mixed_effects_rpe(
                model_out,
                source_roi=source_roi,
                savepath=source_paths['base_dir'],
                event=ANALYSIS['event'],
                fmin=ANALYSIS['fmin'],
                fmax=ANALYSIS['fmax'],
            )
            fig_mixed.get_axes()[0].set_ylim(-2, 2)
            fig_mixed.get_axes()[1].set_ylim(-2, 2)
            sns.despine()
            if fig_mixed is not None:
                plt.show()
            analysis_state.setdefault('model_out', {})[source_roi] = model_out
            if len(model_out.get('stats_df', pd.DataFrame())) > 0:
                stats_file = source_paths['stats_csv']
                model_out['stats_df'].to_csv(stats_file, index=False)
                print('Saved mixed-model stats:', stats_file)
                display(model_out['stats_df'].head())


## Figure 5E

In [ ]:
# ---- Config ----
corr_type = 'spearman'  # 'spearman' or 'pearson'
run_secondary_pearson = True
label_points = True
gc_metric = 'gc_net_z'
freq_band_label = 'theta'  # 4–8 Hz merged GC bundle below

hit_memory_label = 'remembered'
rsa_metric_name = 'HitRSA_cluster_mean'
rsa_target_label = 'ACC-HPC Hit RSA'
rsa_metric_col = f'ACC_HPC_{rsa_metric_name}'

acc_reactivation_metric = 'MeanAccuracy'
acc_reactivation_label = 'ACC Reactivation'
acc_reactivation_col = f'ACC_{acc_reactivation_metric}'
acc_reactivation_analysis_label = 'ACC Reactivation vs ACC-HPC Hit RSA'

rsa_feedback_roi = 'ACC'
rsa_cue_roi = 'HPC'
gc_roi_source = 'HPC'
gc_roi_target = 'ACC'

rsa_group_results_pkl = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/RSA/crossregion/'
    'group_results_ACC_to_HPC.pkl'
)
acc_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/crossdecoding/rpe_category/'
    'ACC_rpe_category_HFA_subject_reinstatement.csv'
)
gc_merged_csv = (
    '/Volumes/T7 Shield/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mvgc_rpe/'
    'notebook_runs/cue_on_Day1_t0p0-1p0_sf50_4-8Hz_lag10_roie86d0eb437/merged_gc_rpe.csv'
)
hpc_theta_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'HPC_memory_outcome_hit_miss_theta_subject_reinstatement.csv'
)
learn_df_csv = '/Volumes/T7 Shield/work/qasims01/MemoryBanditData/EMU/learn_df_RWH.csv'
combined_df_csv = '/Volumes/T7 Shield/work/qasims01/MemoryBanditData/EMU/full_df_RWH.csv'
combined_df_condition = 'Day1'

cluster_alpha = 0.05
cluster_n_permutations = 1000
cluster_seed = 42
cluster_tail = 1

time_windows: Dict[str, Tuple[float, float]] = {
    'cue_full': (0.0, 1.0),
    'cue_first_half': (0.0, 0.5),
    'cue_second_half': (0.5, 1.0),
}

time_window_labels: Dict[str, str] = {
    'cue_full': 'whole cue (0–1 s)',
    'cue_first_half': 'first half (0–0.5 s)',
    'cue_second_half': 'second half (0.5–1 s)',
}

analysis_output_root = Path(
    '/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/'
    'analysis_outputs/acc_hpc_hits_rsa_gc_rpe_theta'
)
figure_output_root = Path(
    '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'
)
analysis_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)

analysis_prefix = (
    f'rpe_category_HFA_ACC_HPC_hits_cluster_mean_vs_{freq_band_label}_{gc_metric}_cue_on'
)
paths: Dict[str, object] = {
    'subject_metric_csv': analysis_output_root / f'{analysis_prefix}_subject_metric.csv',
    'cluster_metadata_csv': analysis_output_root / f'{analysis_prefix}_cluster_metadata.csv',
    'cluster_metadata_json': analysis_output_root / f'{analysis_prefix}_cluster_metadata.json',
    'acc_reactivation_merge_csv': analysis_output_root / f'{analysis_prefix}_acc_reactivation_vs_hit_rsa_subjects.csv',
    'acc_reactivation_results_csv': analysis_output_root / f'{analysis_prefix}_acc_reactivation_vs_hit_rsa_results.csv',
    'subject_covariate_merge_csv': analysis_output_root / f'{analysis_prefix}_subject_covariates_merged_subjects.csv',
    'subject_covariate_results_csv': analysis_output_root / f'{analysis_prefix}_subject_covariates_results.csv',
    'subject_covariate_summary_txt': analysis_output_root / f'{analysis_prefix}_subject_covariates_summary.txt',
    'hpc_theta_merge_csv': analysis_output_root / f'{analysis_prefix}_hpc_theta_mean_accuracy_merged_subjects.csv',
    'hpc_theta_results_csv': analysis_output_root / f'{analysis_prefix}_hpc_theta_mean_accuracy_results.csv',
    'merged_csv': analysis_output_root / f'{analysis_prefix}_merged_subject_condition.csv',
    'results_csv': analysis_output_root / f'{analysis_prefix}_correlation_results.csv',
    'summary_txt': analysis_output_root / f'{analysis_prefix}_summary.txt',
    'scatter_plot_pdfs': {
        wk: figure_output_root / f'grid_scatter_{wk}.pdf'
        for wk in time_windows
    },
    'acc_reactivation_scatter_pdf': figure_output_root / 'acc_reactivation_vs_hit_rsa_scatter.pdf',
    'subject_covariate_scatter_pdf': figure_output_root / 'subject_covariates_scatter.pdf',
    'hpc_theta_scatter_pdf': figure_output_root / 'hpc_theta_mean_accuracy_vs_hit_rsa_scatter.pdf',
    'sliding_corr_pdf': figure_output_root / 'sliding_spearman_gc_vs_hit_rsa.pdf',
    'sliding_corr_csv': analysis_output_root / f'{analysis_prefix}_sliding_spearman_gc_vs_hit_rsa.csv',
}

subject_covariate_predictor_order = [
    'alpha',
    'beta',
    'total_reward',
    'dprime',
    'ACC_NumElectrodes',
    'HPC_NumElectrodes',
]
subject_covariate_predictor_labels = {
    'alpha': 'Alpha',
    'beta': 'Beta',
    'total_reward': 'Total reward',
    'dprime': 'd-prime',
    'ACC_NumElectrodes': 'ACC electrode count',
    'HPC_NumElectrodes': 'HPC electrode count',
}

rpe_cats_sliding: Tuple[str, ...] = ('neg', 'neutral', 'pos')
sliding_window_len_s = 0.2
sliding_step_s = 0.01
sliding_corr_min_n_subjects = 3

expected_conditions: List[Tuple[str, str]] = [
    ('cue', 'neutral'),
    ('cue', 'pos'),
]

print('Analysis output root:', analysis_output_root)
print('Figure output root:', figure_output_root)
print('Analysis prefix:', analysis_prefix)
print('RSA group results PKL:', rsa_group_results_pkl)
print('ACC reinstatement CSV:', acc_reinstatement_csv)
print('GC merged CSV:', gc_merged_csv)
print('HPC theta reinstatement CSV:', hpc_theta_reinstatement_csv)
print('Behavior CSV:', learn_df_csv)
print('Combined CSV:', combined_df_csv, '| condition:', combined_df_condition)
print('Time windows (s):', time_windows)
print('Expected conditions:', expected_conditions)
print('Sliding RPE categories:', rpe_cats_sliding)
print('Sliding corr outputs:', paths['sliding_corr_pdf'])
print('Sliding window len (s):', sliding_window_len_s, '| step (s):', sliding_step_s)


In [ ]:
def assert_required_columns(tbl: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
    missing = [c for c in required_cols if c not in tbl.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def deterministic_inner_join(left: pd.DataFrame, right: pd.DataFrame, keys: List[str]) -> pd.DataFrame:
    out = left.merge(right, on=keys, how='inner')
    return out.sort_values(keys).reset_index(drop=True)


def _extract_corr(res) -> Tuple[float, float]:
    if hasattr(res, 'statistic') and hasattr(res, 'pvalue'):
        return float(res.statistic), float(res.pvalue)
    return float(res[0]), float(res[1])


def run_correlation_analysis(
    x: np.ndarray,
    y: np.ndarray,
    corr_type: str = 'spearman',
    run_secondary_pearson: bool = True,
) -> Dict[str, float]:
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask]
    y_valid = y[mask]
    n = int(len(x_valid))
    if n < 3:
        raise ValueError(f'Need at least 3 valid observations; got {n}.')

    rho_s, p_s = _extract_corr(spearmanr(x_valid, y_valid, nan_policy='omit'))
    if run_secondary_pearson or corr_type.lower() == 'pearson':
        r_p, p_p = _extract_corr(pearsonr(x_valid, y_valid))
    else:
        r_p, p_p = np.nan, np.nan

    corr_type_norm = corr_type.lower()
    if corr_type_norm == 'spearman':
        primary_r, primary_p = rho_s, p_s
    elif corr_type_norm == 'pearson':
        primary_r, primary_p = r_p, p_p
    else:
        raise ValueError(f"Unsupported corr_type: {corr_type}. Use 'spearman' or 'pearson'.")

    return {
        'n': n,
        'spearman_rho': rho_s,
        'spearman_p': p_s,
        'pearson_r': r_p,
        'pearson_p': p_p,
        'primary_corr_type': corr_type_norm,
        'primary_r': primary_r,
        'primary_p': primary_p,
    }


def _fit_and_plot_line(ax, x: np.ndarray, y: np.ndarray) -> None:
    if len(x) < 2 or len(np.unique(x)) < 2:
        return
    coeff = np.polyfit(x, y, 1)
    xfit = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    yfit = np.polyval(coeff, xfit)
    ax.plot(xfit, yfit, color='#222222', linewidth=1.2)


def plot_roi_condition_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    target_label: str,
    gc_col: str,
    conditions: List[Tuple[str, str]],
    output_pdf: Path,
    time_window_key: str,
    time_window_label: str = '',
    label_points: bool = True,
) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    axes = axes.flatten()

    sns.set_style('white')

    for ax, (task_condition, rpe_cat) in zip(axes, conditions):
        panel_df = merged_df[
            (merged_df['task_condition'] == task_condition)
            & (merged_df['rpe_cat'] == rpe_cat)
        ].copy()
        panel_df = panel_df[np.isfinite(panel_df[gc_col]) & np.isfinite(panel_df[target_col])]

        ax.set_title(f'{task_condition} / {rpe_cat}')
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[gc_col].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)

        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, gc_col), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        res_match = results_df[
            (results_df['target_roi'] == target_label)
            & (results_df['time_window'] == time_window_key)
            & (results_df['task_condition'] == task_condition)
            & (results_df['rpe_cat'] == rpe_cat)
        ]
        if len(res_match) > 0:
            rr = res_match.iloc[0]
            ann = f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, p={rr['primary_p']:.3g}, n={int(rr['n'])}"
            ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)

        ax.set_xlabel(gc_col)
        ax.set_ylabel(target_col)

        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    tw = time_window_label or time_window_key
    fig.suptitle(f'{target_label} vs {gc_col} by condition ({tw})', fontsize=14)
    out = Path(output_pdf)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=300)
    plt.close(fig)


def average_gc_in_time_window(
    gc_df: pd.DataFrame,
    tmin: float,
    tmax: float,
    gc_roi_source: str,
    gc_roi_target: str,
    gc_metric: str,
) -> pd.DataFrame:
    m = (
        (gc_df['time'] >= tmin)
        & (gc_df['time'] <= tmax)
        & (gc_df['roi_source'] == gc_roi_source)
        & (gc_df['roi_target'] == gc_roi_target)
    )
    sub = gc_df.loc[m].copy()
    if len(sub) == 0:
        raise ValueError(
            f'No GC rows for {gc_roi_source}->{gc_roi_target} in time [{tmin}, {tmax}].'
        )
    out = (
        sub.groupby(['participant', 'rpe_cat'], as_index=False, dropna=False)
        .agg(**{gc_metric: (gc_metric, 'mean')})
    )
    out = out.rename(columns={'participant': 'Subject'})
    out['task_condition'] = 'cue'
    return out


def build_sliding_spearman_table(
    gc_long: pd.DataFrame,
    target_tbl: pd.DataFrame,
    target_col: str,
    gc_roi_source: str,
    gc_roi_target: str,
    gc_metric: str,
    rpe_cats: Tuple[str, ...],
    window_len_s: float,
    step_s: float,
    min_n: int,
) -> pd.DataFrame:
    g = gc_long.loc[
        (gc_long['roi_source'] == gc_roi_source)
        & (gc_long['roi_target'] == gc_roi_target)
    ].copy()

    times = np.sort(g['time'].unique())
    t_lo = float(np.min(times))
    t_hi = float(np.max(times))
    half = window_len_s / 2.0
    if t_hi - t_lo < window_len_s:
        raise ValueError(
            f'Epoch [{t_lo}, {t_hi}] s is shorter than sliding_window_len_s={window_len_s}.'
        )
    start_c = t_lo + half
    end_c = t_hi - half
    centers = np.arange(start_c, end_c + step_s * 0.5, step_s, dtype=float)
    centers = centers[centers <= end_c + 1e-9]

    target_df = target_tbl[['Subject', target_col]].copy()
    rows = []

    for center in centers:
        tmin = float(center - half)
        tmax = float(center + half)
        m = (g['time'] >= tmin) & (g['time'] <= tmax)
        sub_w = g.loc[m].copy()
        if len(sub_w) == 0:
            continue

        for cat in rpe_cats:
            sc = sub_w[sub_w['rpe_cat'] == cat].copy()
            if len(sc) == 0:
                rows.append({
                    'time': float(center),
                    'window_tmin': tmin,
                    'window_tmax': tmax,
                    'rpe_cat': cat,
                    'spearman_rho': np.nan,
                    'spearman_p': np.nan,
                    'se_rho': np.nan,
                    'mean_gc': np.nan,
                    'sem_mean_gc': np.nan,
                    'n_subjects': 0,
                })
                continue

            grp = (
                sc.groupby('participant', as_index=False, dropna=False)
                .agg(**{gc_metric: (gc_metric, 'mean')})
            )
            grp = grp.rename(columns={'participant': 'Subject'})
            merged = grp.merge(target_df, on='Subject', how='inner')
            x = merged[gc_metric].to_numpy(dtype=float)
            y = merged[target_col].to_numpy(dtype=float)
            mask = np.isfinite(x) & np.isfinite(y)
            x = x[mask]
            y = y[mask]
            n = int(len(x))
            mean_gc = float(np.nanmean(x)) if n > 0 else np.nan
            if n >= 2:
                sem_mean_gc = float(np.std(x, ddof=1) / np.sqrt(n))
            else:
                sem_mean_gc = np.nan
            if n < min_n:
                rows.append({
                    'time': float(center),
                    'window_tmin': tmin,
                    'window_tmax': tmax,
                    'rpe_cat': cat,
                    'spearman_rho': np.nan,
                    'spearman_p': np.nan,
                    'se_rho': np.nan,
                    'mean_gc': mean_gc,
                    'sem_mean_gc': sem_mean_gc,
                    'n_subjects': n,
                })
                continue
            rho, pval = spearmanr(x, y)
            rows.append({
                'time': float(center),
                'window_tmin': tmin,
                'window_tmax': tmax,
                'rpe_cat': cat,
                'spearman_rho': float(rho),
                'spearman_p': float(pval),
                'se_rho': np.nan,
                'mean_gc': mean_gc,
                'sem_mean_gc': sem_mean_gc,
                'n_subjects': n,
            })
    return pd.DataFrame(rows)


def load_roi_electrode_counts(subjects: List[str], base_dir: str) -> pd.DataFrame:
    rows = []
    missing_elec_tables = []
    for subj_id in subjects:
        elec_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df'
        if not os.path.isfile(elec_path):
            missing_elec_tables.append(elec_path)
            continue
        elec_df = pd.read_csv(elec_path)
        region_col = 'salman_region' if 'salman_region' in elec_df.columns else 'region' if 'region' in elec_df.columns else None
        if region_col is None:
            raise ValueError(f'Could not identify ROI column for {subj_id}.')
        rows.append({
            'Subject': subj_id,
            'ACC_NumElectrodes': int(elec_df.loc[elec_df[region_col].astype(str) == 'ACC', 'label'].astype(str).nunique()),
            'HPC_NumElectrodes': int(elec_df.loc[elec_df[region_col].astype(str) == 'HPC', 'label'].astype(str).nunique()),
        })
    if missing_elec_tables:
        print(f'Missing electrode tables for {len(missing_elec_tables)} subject(s).')
    return pd.DataFrame(rows).sort_values('Subject').reset_index(drop=True)


def plot_subject_covariate_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    predictor_order: List[str],
    predictor_labels: Dict[str, str],
    output_pdf: Path,
    label_points: bool = True,
) -> None:
    n_panels = len(predictor_order)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    sns.set_style('white')

    for ax, predictor in zip(axes, predictor_order):
        panel_df = merged_df[np.isfinite(merged_df[predictor]) & np.isfinite(merged_df[target_col])].copy()
        ax.set_title(predictor_labels[predictor])
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[predictor].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)
        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, predictor), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        rr = results_df[results_df['predictor'] == predictor].iloc[0]
        ann = (
            f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, "
            f"p={rr['primary_p']:.3g}, n={int(rr['n'])}"
        )
        ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)
        ax.set_xlabel(predictor_labels[predictor])
        ax.set_ylabel(target_col)
        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    for ax in axes[n_panels:]:
        ax.axis('off')

    fig.suptitle(f'{rsa_target_label} vs subject-level covariates', fontsize=14)
    out = Path(output_pdf)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=300)
    plt.close(fig)
